# ACME: human-readable data

The human-readable part of ACME holds every robot trajectory resampled at **4 Hz**.
For each sample `i` there is an RGB image, a LiDAR point cloud, the robot odometry,
the tracked pedestrian boxes and (for some institutions) a person mask, all taken at
`timestamps[i]`.

```
sampled_data/<institution>/<trajectory>.zip  ->  <trajectory>/
    timestamps.txt  traj_data.csv  tracks.json  speech.txt (some institutions)
    imgs/<i>.jpg  masks/<i>.jpg (some institutions)  pcd/<i>.pcd
sampled_data/extrinsics_intrinsics/<institution>/   camera intrinsics, static transforms
sampled_data/utterances/<institution>_utterances.csv
```

This notebook shows how to load and plot all of it with `acme_dataset`
(`pip install -e .` from the repository root).

## 1. Get the data

Set `DOWNLOAD = True` to fetch one trajectory from Hugging Face (or use `acme-dataset download human_readable -i miraikan -n <trajectory>` in a shell). The whole human-readable part is ~200 GB.

In [ ]:
from pathlib import Path
import os

# Root of the dataset: the folder that contains `ROS2_Bags/`, `sampled_data/`,
# `BEV/` and `acme_tags.xlsx`. Override with the ACME_ROOT environment variable.
DATA_ROOT = Path(os.environ.get("ACME_ROOT", "../ACME-dataset"))

DOWNLOAD = False
if DOWNLOAD:
    from acme_dataset import download, list_remote
    print(list_remote("human_readable"))                        # institutions
    names = list_remote("human_readable", institution="miraikan")
    download("human_readable", "miraikan", names[:1], local_dir=DATA_ROOT)   # ~60 MB, unzipped after download
    download("calibration", "miraikan", local_dir=DATA_ROOT)
    download("tags", local_dir=DATA_ROOT)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from acme_dataset import list_trajectories

trajectories = list_trajectories(DATA_ROOT)
print(f"{len(trajectories)} trajectories")
traj = trajectories[0]
traj, traj.summary()

## 2. Odometry

`traj.odometry` is a DataFrame (odom frame, SI units). The plot shows the path in the frame of the first pose.

In [ ]:
from acme_dataset.viz import plot_odometry

display(traj.odometry.head())
plot_odometry(traj.odometry)
plt.tight_layout()

## 3. Images, pedestrian tracks and person masks

`traj.detections` has one row per (frame, pedestrian); `x, y` is the top-left corner of the box in pixels and `track_id` is consistent within the trajectory.

In [ ]:
from acme_dataset.viz import draw_detections, overlay_mask

display(traj.detections.head())
frames = np.linspace(0, len(traj) - 1, 4).astype(int)
fig, axes = plt.subplots(2, len(frames), figsize=(4 * len(frames), 6))
for col, i in enumerate(frames):
    image = traj.image(i)
    axes[0, col].imshow(draw_detections(image, traj.detections_at(i)))
    axes[0, col].set_title(f"frame {i}: {len(traj.detections_at(i))} pedestrians")
    mask = traj.mask(i)
    axes[1, col].imshow(overlay_mask(image, mask) if mask is not None else image)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()

Each `track_id` can be followed over time, e.g. to get per-pedestrian image tracks:

In [ ]:
per_track = traj.detections.groupby("track_id").agg(
    first_frame=("frame_idx", "min"), last_frame=("frame_idx", "max"), frames=("frame_idx", "size"))
per_track.sort_values("frames", ascending=False).head()

## 4. LiDAR

Point clouds are `(N, 4)` arrays (x, y, z, intensity) in the LiDAR frame.

In [ ]:
from acme_dataset.viz import plot_pointcloud_bev

points = traj.pointcloud(frames[0])
print(points.shape, points[:3])
fig, axes = plt.subplots(1, len(frames), figsize=(4 * len(frames), 4))
for ax, i in zip(axes, frames):
    plot_pointcloud_bev(traj.pointcloud(i), ax=ax, x_range=(-10, 10), y_range=(-10, 10))
    ax.set_title(f"frame {i}")
plt.tight_layout()

For interactive 3D viewing, the `.pcd` files open directly in Open3D, CloudCompare or RViz:

```python
import open3d as o3d
o3d.visualization.draw_geometries([o3d.io.read_point_cloud(str(traj.pointcloud_path(0)))])
```

## 5. Iterating over samples and robot speech

In [ ]:
for frame in traj:
    if frame.index >= 3:
        break
    print(frame.index, round(frame.timestamp, 2), frame.odometry[["position_x", "position_y"]].tolist(),
          len(frame.detections), "pedestrians", frame.pointcloud.shape)

traj.speech  # empty if the robot did not speak in this trajectory

## 6. Calibration

Intrinsics and static transforms per institution, from `sampled_data/extrinsics_intrinsics/` (`acme-dataset download calibration -i <institution>`).

In [ ]:
from acme_dataset.calibration import calibration_files, read_camera_info_txt, read_tf_static_txt

files = calibration_files(DATA_ROOT, traj.institution, traj.name)
print([f.name for f in files])
for f in files:
    if f.name.endswith(("camera_info.txt", "intrinsics.txt")):
        print(f.name, read_camera_info_txt(f)["K"], sep="\n")
    elif f.name == "_tf_static.txt":
        display(read_tf_static_txt(f).head())

## 7. Scenario tags

`acme_tags.xlsx` labels every trajectory with location and pedestrian scenario tags.

In [ ]:
from acme_dataset import filter_by_tags, load_scenario_tags, tag_counts, tags_for

tags_path = DATA_ROOT / "acme_tags.xlsx"
if tags_path.exists():
    tags = load_scenario_tags(tags_path)
    print("tags of this trajectory:", tags_for(tags, traj.name))
    display(tag_counts(tags, by_institution=True))
    display(filter_by_tags(tags, all_of=["narrow_corridors", "against_traffic"]).head())

## 8. Dataset statistics

The same numbers are available from the shell: `acme-dataset stats <DATA_ROOT> --out stats.csv`.

In [ ]:
from acme_dataset import dataset_stats, summarize

stats = dataset_stats(DATA_ROOT)
display(stats.head())
summarize(stats)